# Statistical Analysis: Final Experiment — Does the Tail Help Reorientation?

**Goal:** Test if the tail (With Tail vs No Tail) affects reorientation performance in the final drop-test experiment.

**Data:** `report_final.csv`, produced by `data_analysis_final.py` from `telemetry/Final_Experiment/`. Fall-duration/distance filtering, the `0°-pitch`/`180°-roll` de-dup, and the spare-rep exclusion are already applied upstream — this notebook starts from the cleaned per-drop report.

**Problem:** Initial conditions (angles and velocities at release) vary randomly due to drop rig imprecision.

**Solution:** OLS regression controlling for trial condition and initial-condition variability (mirrors `stats_final.py`, expanded here for interactive exploration).

**Measurement:** Front IMU only.

**Conditions:** 3 roll magnitudes (`r45`, `r90`, `r180`) + 3 combined roll180+pitch conditions (`r180_p15`, `r180_p30`, `r180_p45`).

**Outcomes:** |Roll angle|, |Roll rate|, |Pitch angle|, |Pitch rate| at 2.5 m — absolute values, since deviation from upright in either direction is equally bad.

No cell outputs are pre-computed in this file — run all cells to populate plots/output.

**v1 note:** this version applies fixes from a methodology review (robust SEs, corrected multiple comparisons, a blocking check, precision-framed power). See `stats_final_v2.ipynb` for additional deeper checks (permutation tests, TOST equivalence, variance/spread analysis, covariate centering) that go beyond patching this notebook's structure.

---


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

# Condition ordering used throughout (roll sweep, then pitch sweep) instead of alphabetical
ROLL_CONDITION_ORDER  = ['r45', 'r90', 'r180']
PITCH_CONDITION_ORDER = ['r180_p15', 'r180_p30', 'r180_p45']
CONDITION_ORDER       = ROLL_CONDITION_ORDER + PITCH_CONDITION_ORDER

MORPHOLOGIES = ['With Tail', 'No Tail']

# Bonferroni correction used in Section 6: the per-trial loop there tests all 4 outcomes
# (roll angle, roll rate, pitch angle, pitch rate) at each of the 6 trial conditions —
# 6 x 4 = 24 independent t-tests, so that's the correction factor (not the 3x2+3x2=12 pairing
# used in the older stats_mbc.py, which only tested each condition against its "native" outcome).
N_COMPARISONS = 24


## 1. Load Data and Create Variables

In [ ]:
df = pd.read_csv('report_final.csv', dtype={'date': str})

print(f"Total drops: {len(df)}")
print(f"Morphologies: {df['morphology'].unique()}")
print(f"Trials: {df['trial'].unique()}")
print(f"\nDrops per morphology x trial:")
print(pd.crosstab(df['morphology'], df['trial'])[CONDITION_ORDER])


In [ ]:
# Front IMU only, with angle wrapping fix
#
# IMPORTANT: F_roll values can exceed ±180° due to np.unwrap() in the processing pipeline.
# Taking abs() of 355° would give 355°, but physically that's only 5° from upright.
# We use shortest angular distance instead: wrap to [-180, 180], then abs().

def angular_distance(angle):
    '''Shortest angular distance from 0. Handles wrapped angles correctly.'''
    return np.abs(((angle + 180) % 360) - 180)

def wrap_angle(angle):
    '''Wrap signed angle to [-180, 180].'''
    return ((angle + 180) % 360) - 180

# Outcomes: shortest angular distance from upright
df['abs_roll_2p5m']      = angular_distance(df['F_roll_2p5m'])
df['abs_pitch_2p5m']     = angular_distance(df['F_pitch_2p5m'])
df['abs_rollrate_2p5m']  = df['F_rollrate_2p5m'].abs()  # rates don't wrap
df['abs_pitchrate_2p5m'] = df['F_pitchrate_2p5m'].abs()

# Initial conditions: wrap angles, keep rates signed
df['roll_initial']      = wrap_angle(df['F_roll_initial'])
df['pitch_initial']     = wrap_angle(df['F_pitch_initial'])
df['rollrate_initial']  = df['F_rollrate_initial']
df['pitchrate_initial'] = df['F_pitchrate_initial']

print(f"Roll outcome range: [{df['abs_roll_2p5m'].min():.1f}, {df['abs_roll_2p5m'].max():.1f}] (should be ≤ 180)")
print(f"Pitch outcome range: [{df['abs_pitch_2p5m'].min():.1f}, {df['abs_pitch_2p5m'].max():.1f}] (should be ≤ 180)")
df[['morphology', 'trial', 'abs_roll_2p5m', 'abs_rollrate_2p5m', 'abs_pitch_2p5m', 'abs_pitchrate_2p5m']].head(10)


## 2. Diagnostic Checks: Confounds and Blocking

**Question A — within-condition imbalance:** does tail presence systematically affect initial conditions, controlling for trial?

**Method:** Two-way ANOVA: `Initial_Condition ~ Morphology + Trial`. p > 0.05 → no detectable imbalance; p < 0.05 → flagged.

Four non-significant tests is *absence of evidence*, not evidence of absence — treat "no detectable imbalance" as a weaker claim than "confirmed random," especially for any variable close to the 0.05 threshold. The model in Section 4 adjusts for these covariates regardless, so causal interpretation doesn't hinge entirely on this check passing.

**Question B — collection-date blocking:** the two morphologies were not collected in a randomized interleaved order — they're unevenly split across the three collection dates. If anything drifted day-to-day (battery, rig calibration, joint wear), it could load onto the morphology coefficient. This is checked directly below by adding `C(date)` to the model and seeing whether the tail coefficient moves.

In [ ]:
# Visualize initial conditions by trial and morphology
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

initial_vars = [
    ('roll_initial',      'Initial Roll (deg)'),
    ('pitch_initial',     'Initial Pitch (deg)'),
    ('rollrate_initial',  'Initial Roll Rate (deg/s)'),
    ('pitchrate_initial', 'Initial Pitch Rate (deg/s)')
]

for idx, (var, label) in enumerate(initial_vars):
    sns.boxplot(data=df, x='trial', y=var, hue='morphology', ax=axes[idx], order=CONDITION_ORDER)
    axes[idx].set_title(label)
    axes[idx].set_xlabel('Trial')
    axes[idx].set_ylabel(label)
    axes[idx].tick_params(axis='x', rotation=45)

plt.suptitle('Initial Conditions by Trial and Morphology\n(Colors should overlap randomly — if consistently separated, that is a confound)', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Two-way ANOVA: Initial_Condition ~ Morphology + Trial
print("="*80)
print("DIAGNOSTIC A: Does tail presence affect initial conditions (controlling for trial)?")
print("="*80)

diagnostic_results = []
confound_flags = {}  # var -> bool, reused in Section 4/6 to flag caution on ANY outcome, not just the matched one

for var, label in initial_vars:
    model = smf.ols(f"{var} ~ C(morphology) + C(trial)", data=df).fit()
    anova_table = anova_lm(model, typ=2)

    morph_f = anova_table.loc['C(morphology)', 'F']
    morph_p = anova_table.loc['C(morphology)', 'PR(>F)']
    is_confounded = morph_p < 0.05
    result  = "✅ no detectable imbalance" if not is_confounded else "⚠️ CONFOUNDED"

    print(f"\n{label}")
    print(f"  Morphology effect: F = {morph_f:.3f}, p = {morph_p:.4f} → {result}")

    diagnostic_results.append({'Variable': label, 'F': morph_f, 'p-value': morph_p, 'Result': result})
    confound_flags[var] = is_confounded

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(pd.DataFrame(diagnostic_results).to_string(index=False))
print("\nNote: roll and pitch are mechanically coupled (Kane-Scher effect), so a confound in one")
print("initial variable can leak into an outcome it isn't nominally paired with. Section 4 checks")
print("outcome models against ALL flagged initial variables, not only the matching one.")


In [ ]:
# DIAGNOSTIC B: collection-date blocking check
print("="*80)
print("DIAGNOSTIC B: Is morphology confounded with collection date?")
print("="*80)
print(pd.crosstab(df['date'], df['morphology']))
print()

MORPH_TERM  = 'C(morphology, Treatment(reference="No Tail"))'
MORPH_PARAM = f'{MORPH_TERM}[T.With Tail]'
FORMULA_BASE = f"{MORPH_TERM} + C(trial) + roll_initial + pitch_initial + rollrate_initial + pitchrate_initial"

print("Sensitivity check: does adding C(date) move the tail coefficient?")
print("-" * 80)
for var, label in [('abs_roll_2p5m', '|Roll Angle|'), ('abs_rollrate_2p5m', '|Roll Rate|')]:
    m_base = smf.ols(f"{var} ~ {FORMULA_BASE}", data=df).fit()
    m_date = smf.ols(f"{var} ~ {FORMULA_BASE} + C(date)", data=df).fit()
    c_base, c_date = m_base.params[MORPH_PARAM], m_date.params[MORPH_PARAM]
    pct_change = (c_date - c_base) / abs(c_base) * 100

    ci_base = m_base.conf_int().loc[MORPH_PARAM]
    ci_date = m_date.conf_int().loc[MORPH_PARAM]
    ci_overlap = not (ci_date[1] < ci_base[0] or ci_date[0] > ci_base[1])

    print(f"{label}:")
    print(f"  without C(date): coef={c_base:+.2f}, 95% CI=[{ci_base[0]:.2f}, {ci_base[1]:.2f}], p={m_base.pvalues[MORPH_PARAM]:.4f}")
    print(f"  with C(date)   : coef={c_date:+.2f}, 95% CI=[{ci_date[0]:.2f}, {ci_date[1]:.2f}], p={m_date.pvalues[MORPH_PARAM]:.4f}")
    print(f"  → moved {pct_change:+.0f}% ({'same direction' if np.sign(c_base) == np.sign(c_date) else 'SIGN FLIPS — investigate'}, CIs {'overlap' if ci_overlap else 'do NOT overlap'})")
    print(f"  → date is doing real work in the model; this is a substantial adjustment, not confirmation of stability.")
print()
print("Caveat: C(date) is highly collinear with C(trial) here (the pitch sweep only ran on 08/17,")
print("r45/r90 only on 08/10 and 08/16), so no adjustment can cleanly separate 'date effect' from")
print("'trial effect' — this is a genuine design limitation (imbalanced blocking), not something")
print("this sensitivity check resolves. It belongs in the limitations section as a design flaw.")


## 3. Descriptive Statistics: Outcomes at 2.5m

In [ ]:
outcomes = [
    ('abs_roll_2p5m',      '|Roll Angle| at 2.5m (deg)'),
    ('abs_rollrate_2p5m',  '|Roll Rate| at 2.5m (deg/s)'),
    ('abs_pitch_2p5m',     '|Pitch Angle| at 2.5m (deg)'),
    ('abs_pitchrate_2p5m', '|Pitch Rate| at 2.5m (deg/s)')
]

for var, label in outcomes:
    print(f"\n{label}:")
    print(df.groupby('morphology')[var].describe()[['mean', 'std', 'min', 'max', 'count']])


In [ ]:
# Visualize outcomes by trial and morphology
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (var, label) in enumerate(outcomes):
    sns.boxplot(data=df, x='trial', y=var, hue='morphology', ax=axes[idx], order=CONDITION_ORDER)
    axes[idx].set_title(label)
    axes[idx].set_xlabel('Trial')
    axes[idx].set_ylabel(label)
    axes[idx].tick_params(axis='x', rotation=45)

plt.suptitle('Outcomes at 2.5m by Trial and Morphology', y=1.02)
plt.tight_layout()
plt.show()


## 4. Main Analysis: Linear Model

**Model:** `|Outcome| ~ Morphology + Trial + Initial_Roll + Initial_Pitch + Initial_RollRate + Initial_PitchRate`

- Controls for trial condition (intentional design) and initial condition noise (drop rig)
- `No Tail` is set as the reference level, so **the coefficient for `C(morphology, Treatment(reference="No Tail"))[T.With Tail]`** reads directly as "how much With Tail differs from No Tail" — positive = tail has *more* deviation (worse), negative = tail has *less* deviation (better)
- Reported alongside the standard (nonrobust) SEs are HC3 heteroscedasticity-robust SEs, since Section 8 finds non-normal, heteroscedastic residuals on 3 of 4 outcomes — if HC3 and nonrobust disagree substantially, trust HC3
- A condition-number warning shows up in some model summaries below (~6,000). This comes from `pitch_initial` being nearly collinear with the trial dummies — the three pitch conditions are *defined* by their initial pitch offset, so there's little within-condition variation left to separate the two. It inflates the *trial* coefficients' standard errors, not the morphology coefficient (which lives in a different, well-conditioned subspace); `stats_final_v2.ipynb` demonstrates this directly by re-fitting with within-trial-centered covariates, which reproduces an identical morphology coefficient at a much lower condition number.

In [ ]:
def print_effect(model, label, matched_initial_var=None):
    coef = model.params[MORPH_PARAM]
    pval = model.pvalues[MORPH_PARAM]
    ci   = model.conf_int().loc[MORPH_PARAM]

    model_hc3 = smf.ols(model.model.formula, data=df).fit(cov_type='HC3')
    pval_hc3  = model_hc3.pvalues[MORPH_PARAM]

    print(f"Tail effect on {label}")
    print(f"  Coefficient        : {coef:+.2f} (With Tail vs No Tail)")
    print(f"  95% CI             : [{ci[0]:.2f}, {ci[1]:.2f}]")
    print(f"  p-value (nonrobust): {pval:.4f}")
    print(f"  p-value (HC3)      : {pval_hc3:.4f}")
    print(f"  R²                 : {model.rsquared:.3f}")
    print(f"  {'✅ SIGNIFICANT' if pval < 0.05 else '⚠️ NOT SIGNIFICANT'} (nonrobust); {'✅ SIGNIFICANT' if pval_hc3 < 0.05 else '⚠️ NOT SIGNIFICANT'} (HC3)")

    flagged = [v for v in confound_flags if confound_flags[v]]
    if flagged:
        matched_note = f" (matched: {matched_initial_var})" if matched_initial_var in flagged else ""
        print(f"  ⚠️ Confounded initial variable(s) in Diagnostic A: {flagged}{matched_note} — interpret with caution")


### 4.1 |Impact Roll Angle|

In [ ]:
formula = f"abs_roll_2p5m ~ {FORMULA_BASE}"
model_roll_angle = smf.ols(formula, data=df).fit()
print(model_roll_angle.summary())


In [ ]:
print_effect(model_roll_angle, '|Impact Roll Angle|', matched_initial_var='roll_initial')

### 4.2 |Impact Roll Rate|

In [ ]:
formula = f"abs_rollrate_2p5m ~ {FORMULA_BASE}"
model_roll_rate = smf.ols(formula, data=df).fit()
print(model_roll_rate.summary())


In [ ]:
print_effect(model_roll_rate, '|Impact Roll Rate|', matched_initial_var='rollrate_initial')

### 4.3 |Impact Pitch Angle|

In [ ]:
formula = f"abs_pitch_2p5m ~ {FORMULA_BASE}"
model_pitch_angle = smf.ols(formula, data=df).fit()
print(model_pitch_angle.summary())


In [ ]:
print_effect(model_pitch_angle, '|Impact Pitch Angle|', matched_initial_var='pitch_initial')

### 4.4 |Impact Pitch Rate|

In [ ]:
formula = f"abs_pitchrate_2p5m ~ {FORMULA_BASE}"
model_pitch_rate = smf.ols(formula, data=df).fit()
print(model_pitch_rate.summary())


In [ ]:
print_effect(model_pitch_rate, '|Impact Pitch Rate|', matched_initial_var='pitchrate_initial')

## 5. Summary of All Results

In [ ]:
models = [
    (model_roll_angle,  '|Roll Angle| (deg)',   'roll_initial'),
    (model_roll_rate,   '|Roll Rate| (deg/s)',  'rollrate_initial'),
    (model_pitch_angle, '|Pitch Angle| (deg)',  'pitch_initial'),
    (model_pitch_rate,  '|Pitch Rate| (deg/s)', 'pitchrate_initial'),
]

results_summary = []
for model, outcome, matched_var in models:
    coef = model.params[MORPH_PARAM]
    pval = model.pvalues[MORPH_PARAM]
    ci   = model.conf_int().loc[MORPH_PARAM]

    # HC3 is the SE we said to trust in Section 4 (non-normal residuals) — the CI and significance
    # flag we lead with below should come from the same fit, not the nonrobust one.
    model_hc3 = smf.ols(model.model.formula, data=df).fit(cov_type='HC3')
    ci_hc3    = model_hc3.conf_int().loc[MORPH_PARAM]
    pval_hc3  = model_hc3.pvalues[MORPH_PARAM]

    sig  = '✅' if pval_hc3 < 0.05 else '⚠️'
    flagged = [v for v in confound_flags if confound_flags[v]]
    note = f'(caution: {", ".join(flagged)} confounded)' if flagged else ''
    results_summary.append({
        'Outcome': outcome,
        'Coef': f"{coef:+.2f}",
        '95% CI (nonrobust)': f"[{ci[0]:.2f}, {ci[1]:.2f}]",
        '95% CI (HC3)': f"[{ci_hc3[0]:.2f}, {ci_hc3[1]:.2f}]",
        'p-value (nonrobust)': f"{pval:.4f}",
        'p-value (HC3)': f"{pval_hc3:.4f}",
        'Sig': sig,
        'R²': f"{model.rsquared:.3f}",
        'Note': note
    })

print("="*110)
print("SUMMARY: Tail Effects (With Tail vs No Tail)")
print("Controlling for: Trial condition + Initial conditions | Front IMU only | Absolute values")
print("="*110)
print(pd.DataFrame(results_summary).to_string(index=False))
print("\nNegative coefficient = With Tail has lower absolute deviation than No Tail (tail helps)")
print("Positive coefficient = With Tail has higher absolute deviation than No Tail (tail hurts)")

roll_angle_row = results_summary[0]  # |Roll Angle| (deg)
print(f"\nRead the HC3 CIs, not just the p-values — that's the SE we're trusting given the non-normal")
print(f"residuals. E.g. roll angle's HC3 CI is {roll_angle_row['95% CI (HC3)']} (nonrobust was")
print(f"{roll_angle_row['95% CI (nonrobust)']}): consistent with a tail benefit inside that range,")
print(f"and rules out a tail HARM larger than the CI's upper bound.")


## 6. Per-Trial Analysis

The overall analysis pools trials and may wash out effects that only show up in specific conditions — but running 6 conditions × 2 tests (t-test + MWU) × 4 outcomes is 48 comparisons, and at N≈5–7 per group per trial each individual test is very low powered. Two complementary ways to handle this:

1. **Formal interaction test**: does the tail's effect actually differ across trial conditions? A `morphology × trial` interaction term in the main model tests this directly, with much better power than 6 separate n≈6 comparisons.
2. **Per-trial t-tests, Bonferroni-corrected**: for exploratory/descriptive purposes, shown below with `α_adjusted = 0.05 / 12` (matching `stats_final.py`) rather than raw p-values — the version that gets plotted with significance markers.

> ⚠️ **Low power either way:** N≈5–7 per group per trial. Non-significance ≠ no effect.

In [ ]:
# Formal test: does the tail effect vary by trial condition?
# Bonferroni-corrected across the 4 outcomes, since we're running this test 4 times.
print("Interaction test: morphology × trial (Bonferroni-corrected across 4 outcomes)")
print("=" * 80)

interaction_ps = {}
for var, label in outcomes:
    formula_int = f"{var} ~ {MORPH_TERM} * C(trial) + roll_initial + pitch_initial + rollrate_initial + pitchrate_initial"
    m_int = smf.ols(formula_int, data=df).fit()
    anova_int = anova_lm(m_int, typ=2)
    int_rows = [r for r in anova_int.index if ':' in r]
    p_int = anova_int.loc[int_rows[0], 'PR(>F)'] if int_rows else np.nan
    p_int_adj = min(p_int * len(outcomes), 1.0)
    interaction_ps[label] = (p_int, p_int_adj)
    sig_raw = '✅ (raw)' if p_int < 0.05 else ''
    sig_adj = '✅ (Bonferroni)' if p_int_adj < 0.05 else ''
    print(f"  {label:30s}: raw p = {p_int:.4f} {sig_raw:15s} adjusted p = {p_int_adj:.4f} {sig_adj}")
    print(f"    (model fits {len(m_int.params)} params on {int(m_int.df_resid)} residual df, n={int(m_int.nobs)})")

any_raw_sig = any(p < 0.05 for p, _ in interaction_ps.values())
any_adj_sig = any(p_adj < 0.05 for _, p_adj in interaction_ps.values())

print()
if any_raw_sig and not any_adj_sig:
    flagged = [l for l, (p, _) in interaction_ps.items() if p < 0.05]
    print(f"⚠️→✅ {flagged} crossed the raw p<0.05 threshold but NOT the Bonferroni-adjusted one.")
    print("  With 4 tests run, one crossing raw 0.05 by chance is close to the expected false-positive")
    print("  rate — treat it as noise rather than a real trial-specific tail effect, especially given")
    print("  the interaction model's parameter count relative to residual df (see counts above).")
elif any_adj_sig:
    print("⚠️ At least one interaction survives Bonferroni correction — the tail effect genuinely")
    print("  differs by trial condition here. Pooling into a single overall coefficient (Section 5)")
    print("  would misrepresent that outcome; report it per-trial instead.")
else:
    print("✅ No interaction reaches even the raw 0.05 threshold — pooling trials into the overall")
    print("  Section 5 coefficient is well supported as the better-powered summary.")


In [ ]:
ALPHA_CORRECTED = 0.05 / N_COMPARISONS

per_trial_results = []
for trial in CONDITION_ORDER:
    dft = df[df['trial'] == trial]
    row = {'Trial': trial}
    for var, label in outcomes:
        a = dft[dft['morphology'] == 'With Tail'][var]
        b = dft[dft['morphology'] == 'No Tail'][var]
        diff = a.mean() - b.mean()  # With Tail minus No Tail
        _, p_t = stats.ttest_ind(a, b)
        _, p_u = stats.mannwhitneyu(a, b, alternative='two-sided')
        p_t_adj = min(p_t * N_COMPARISONS, 1.0)
        row[f'{label} diff']       = round(diff, 1)
        row[f'{label} t p_adj']    = f"{p_t_adj:.3f}{'  ✅' if p_t_adj < 0.05 else ''}"
        row[f'{label} MWU p_raw']  = f"{p_u:.3f}"
    per_trial_results.append(row)

per_trial_df = pd.DataFrame(per_trial_results).set_index('Trial')

print('=' * 110)
print('PER-TRIAL TAIL EFFECTS (With Tail vs No Tail)')
print(f'Bonferroni-corrected t-test (alpha_adj={ALPHA_CORRECTED:.4f}) + raw Mann-Whitney U | N≈5-7/group/trial')
print('=' * 110)
print(per_trial_df.to_string())
print('\n✅ = p_adj < 0.05 | Positive diff = With Tail has more deviation')


In [ ]:
# Heatmaps: mean difference and Bonferroni-adjusted p-values
diff_data, pval_data = [], []
for trial in CONDITION_ORDER:
    dft = df[df['trial'] == trial]
    diff_row, pval_row = [], []
    for var, _ in outcomes:
        a = dft[dft['morphology'] == 'With Tail'][var]
        b = dft[dft['morphology'] == 'No Tail'][var]
        diff_row.append(a.mean() - b.mean())
        _, p = stats.ttest_ind(a, b)
        pval_row.append(min(p * N_COMPARISONS, 1.0))
    diff_data.append(diff_row); pval_data.append(pval_row)

col_labels = [l for _, l in outcomes]
diff_df = pd.DataFrame(diff_data, index=CONDITION_ORDER, columns=col_labels)
pval_df = pd.DataFrame(pval_data, index=CONDITION_ORDER, columns=col_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.heatmap(diff_df, annot=True, fmt='.1f', center=0, cmap='RdBu_r', ax=axes[0], linewidths=0.5)
axes[0].set_title('Mean difference (With Tail minus No Tail)\nRed = tail has more deviation, Blue = tail has less deviation')
sns.heatmap(pval_df, annot=True, fmt='.3f', vmin=0, vmax=1.0, cmap='RdYlGn_r', ax=axes[1], linewidths=0.5)
axes[1].set_title('Bonferroni-adjusted p-values (t-test)\n(Green = significant)')
plt.tight_layout()
plt.show()


In [ ]:
# Box plots per trial per outcome, with Bonferroni-corrected significance markers
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (var, label) in enumerate(outcomes):
    ax = axes[idx]
    sns.boxplot(data=df, x='trial', y=var, hue='morphology', ax=ax, order=CONDITION_ORDER)
    sns.stripplot(data=df, x='trial', y=var, hue='morphology', ax=ax,
                  dodge=True, alpha=0.5, size=4, order=CONDITION_ORDER, legend=False)

    ymax = df[var].max()
    for tidx, trial in enumerate(CONDITION_ORDER):
        dft = df[df['trial'] == trial]
        a = dft[dft['morphology'] == 'With Tail'][var]
        b = dft[dft['morphology'] == 'No Tail'][var]
        _, p = stats.ttest_ind(a, b)
        p_adj = min(p * N_COMPARISONS, 1.0)
        if p_adj < 0.05:
            ax.text(tidx, ymax * 1.05, '*', ha='center', fontsize=20, color='red', fontweight='bold')

    ax.set_title(label)
    ax.set_xlabel('Trial')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Outcomes by Trial and Morphology (* = Bonferroni-adjusted t-test p < 0.05)', y=1.00)
plt.tight_layout()
plt.show()


## 7. Effect Size Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

outcomes_list = [r['Outcome'] for r in results_summary]
coeffs    = [float(r['Coef']) for r in results_summary]
ci_lower  = [float(r['95% CI (HC3)'].split(',')[0].strip('[')) for r in results_summary]
ci_upper  = [float(r['95% CI (HC3)'].split(',')[1].strip(']')) for r in results_summary]
pvals     = [float(r['p-value (HC3)']) for r in results_summary]
confounds = [r['Note'] != '' for r in results_summary]

colors = []
for sig, conf in zip([p < 0.05 for p in pvals], confounds):
    if sig and not conf:
        colors.append('green')
    elif sig and conf:
        colors.append('orange')
    else:
        colors.append('gray')

y_pos = np.arange(len(outcomes_list))
xerr  = [np.array(coeffs) - np.array(ci_lower), np.array(ci_upper) - np.array(coeffs)]

ax.barh(y_pos, coeffs, xerr=xerr, color=colors, alpha=0.7, capsize=5)
ax.set_yticks(y_pos)
ax.set_yticklabels(outcomes_list)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Coefficient (With Tail vs No Tail)', fontsize=12)
ax.set_title('Tail Effect Sizes with 95% HC3 CI\n(Controlling for Trial + Initial Conditions, Front IMU, Absolute Values)', fontsize=13)
ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green',  alpha=0.7, label='Significant (p < 0.05)'),
    Patch(facecolor='orange', alpha=0.7, label='Significant but confounded'),
    Patch(facecolor='gray',   alpha=0.7, label='Not Significant')
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()


## 8. Assumption Checks

In [ ]:
print("Normality of Residuals (Shapiro-Wilk):")
print("="*60)
for model, outcome, _ in models:
    stat, p = stats.shapiro(model.resid)
    result = "✅ Normal" if p >= 0.05 else "⚠️ Non-normal"
    print(f"{outcome:25s}: W={stat:.4f}, p={p:.4f} → {result}")

print("\nNon-normal residuals are why Section 4 reports HC3 robust SEs alongside nonrobust ones.")
print("Shapiro-Wilk is also underpowered at this N — check the residual plots below too.")


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 14))

for idx, (model, outcome, _) in enumerate(models):
    axes[idx, 0].scatter(model.fittedvalues, model.resid, alpha=0.5)
    axes[idx, 0].axhline(0, color='red', linestyle='--')
    axes[idx, 0].set_xlabel('Fitted Values')
    axes[idx, 0].set_ylabel('Residuals')
    axes[idx, 0].set_title(f'{outcome}: Residual Plot')
    axes[idx, 0].grid(alpha=0.3)

    axes[idx, 1].hist(model.resid, bins=15, edgecolor='black', alpha=0.7)
    axes[idx, 1].set_xlabel('Residuals')
    axes[idx, 1].set_ylabel('Frequency')
    axes[idx, 1].set_title(f'{outcome}: Residual Distribution')
    axes[idx, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Interpretation Guide

### How to Read Results:

| Color | Meaning |
|-------|--------|
| 🟢 Green | Significant and clean |
| 🟠 Orange | Significant but a Diagnostic-A initial variable was flagged confounded — interpret with caution |
| ⚫ Gray | Not significant |

**Coefficient interpretation:**
- Negative → With Tail has **less** deviation than No Tail (tail helps)
- Positive → With Tail has **more** deviation than No Tail (tail hurts)
- Primary interest: roll outcomes (the tail actuator primarily affects roll). Pitch outcomes are secondary.

**Overall vs per-trial vs interaction:**
- Overall model (Section 4/5) pools all 6 trial conditions → most power, the headline number
- Interaction test (Section 6) asks whether that pooling is defensible in the first place
- Per-trial tests (Section 6) are exploratory/descriptive only, Bonferroni-corrected — treat as a diagnostic for *where* an effect might concentrate, not a standalone claim

**On "no significant effect":** prefer stating the CI directly (e.g. "consistent with a tail benefit up to ~10°, rules out tail harm beyond ~1.6°") over "not significant," which collapses a range of plausible effects into a binary.

---

### Statistical Notes:
1. **Angle wrapping**: `np.unwrap()` in the processing pipeline can produce values like 355° or -377°. This notebook wraps to [-180, 180] first, *then* takes abs — shortest angular distance from upright.
2. **No cluster-robust SE**: only 6 trial clusters — below the ~30 needed for cluster SE to be reliable. HC3 heteroscedasticity-robust SEs are reported instead (Section 4).
3. **Per-trial tests use no covariates**: with N≈5–7 per group, adding 4 covariates would overfit.
4. **Achieved-power-at-observed-effect (old Section 10) is a monotone function of the p-value** and shouldn't be read as independent evidence — see Section 10 below for the reframed version.

### Known Limitations:
1. Confound status for each initial variable is determined *live* by Diagnostic A above — `confound_flags` propagates into every outcome section, not just the nominally matched one, since roll/pitch are mechanically coupled.
2. Morphology is not randomly interleaved across collection dates — see Diagnostic B. The sensitivity check is reassuring but coarse (date is collinear with trial).
3. Low power per trial (N≈5–7 per group) — non-significance ≠ no effect (Section 10).
4. `r90` has an asymmetric N across morphologies (one With-Tail drop was filtered for falling short of the 2.5 m/0.7 s threshold).

## 10. Power and Precision

Since no significant tail effect was found in the overall model, this section asks two different questions that are easy to conflate:

1. **Design-based MDE**: given this sample size, what effect size *could* this design have reliably caught at 80% power? This doesn't depend on what we observed — it's a property of N alone, and it's the legitimate way to talk about "was this study powered enough."
2. **Achieved power at the observed effect**: shown for reference only. This number is mathematically a monotone transform of the p-value you already have — it cannot tell you anything the p-value didn't, and post-hoc power is a well-known statistical anti-pattern (see Hoenig & Heisey, 2001, *The Abuse of Power*). Don't cite it as independent support for "underpowered"; the MDE comparison already makes that case honestly.

In [ ]:
from statsmodels.stats.power import TTestIndPower

power_analysis = TTestIndPower()
alpha = 0.05
power = 0.80

n_overall   = min((df['morphology'] == 'With Tail').sum(), (df['morphology'] == 'No Tail').sum())
n_per_trial = 6   # typical per-group N per trial (varies 5-7, see Section 1 crosstab)

mde_overall   = power_analysis.solve_power(nobs1=n_overall,   alpha=alpha, power=power)
mde_per_trial = power_analysis.solve_power(nobs1=n_per_trial, alpha=alpha, power=power)

print("Minimum Detectable Effect Size (Cohen's d) at 80% power, α=0.05 — design-based, not post-hoc")
print("=" * 70)
print(f"  Overall analysis   (N={n_overall} per group): d = {mde_overall:.3f}")
print(f"  Per-trial analysis (N≈{n_per_trial} per group): d = {mde_per_trial:.3f}")
print(f"Cohen's d benchmarks: small=0.2, medium=0.5, large=0.8")


In [ ]:
# Effect size computed from the ADJUSTED model (coef / sqrt(MSE)), consistent with the model's own
# p-value — not from raw pooled group stats, which would answer a slightly different question.
print("Model-implied effect size (coefficient / sqrt(residual MSE)) — for reference only, see caveat above")
print("=" * 80)

observed_ds_signed = {}
for model, label, _ in models:
    d = model.params[MORPH_PARAM] / np.sqrt(model.mse_resid)
    observed_ds_signed[label] = d
    achieved_power = power_analysis.solve_power(effect_size=abs(d), nobs1=n_overall, alpha=alpha)
    print(f"  {label:25s}: d = {d:+.3f}, achieved power = {achieved_power:.3f}  (informational — see caveat above)")

print(f"\nMDE (overall, 80% power): d = {mde_overall:.3f}")
print("This says the design could reliably catch effects at or above this size; observed effects")
print("below it don't confirm 'no effect', just that this N can't resolve effects that small.")


In [ ]:
# ── Power curve plot ─────────────────────────────────────────────────────────
effect_sizes = np.linspace(0.1, 2.5, 200)

power_overall   = [power_analysis.solve_power(effect_size=d, nobs1=n_overall,   alpha=alpha) for d in effect_sizes]
power_per_trial = [power_analysis.solve_power(effect_size=d, nobs1=n_per_trial, alpha=alpha) for d in effect_sizes]

observed_ds = {label: abs(d) for label, d in observed_ds_signed.items()}

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(effect_sizes, power_overall,   label=f'Overall (N={n_overall}/group)',   color='steelblue', linewidth=2)
ax.plot(effect_sizes, power_per_trial, label=f'Per-trial (N≈{n_per_trial}/group)', color='tomato',    linewidth=2)
ax.axhline(0.80, color='black', linestyle='--', linewidth=1, label='80% power target')
ax.axvline(mde_overall,   color='steelblue', linestyle=':', linewidth=1)
ax.axvline(mde_per_trial, color='tomato',    linestyle=':', linewidth=1)

colors = ['green', 'orange', 'purple', 'brown']
for (label, d), col in zip(observed_ds.items(), colors):
    ax.axvline(d, color=col, linestyle='-.', linewidth=1, alpha=0.7, label=f'{label}: d={d:.2f}')

ax.set_xlabel("Cohen's d (effect size)", fontsize=12)
ax.set_ylabel("Statistical Power", fontsize=12)
ax.set_title("Power Curves: What Effect Sizes Can We Detect? (design-based MDE, not post-hoc)", fontsize=12)
ax.set_xlim(0, 2.5)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Summary:")
print(f"  Overall analysis is powered for d >= {mde_overall:.2f}")
print(f"  Per-trial analysis is powered for d >= {mde_per_trial:.2f} (very large effect)")
print(f"  Model-implied effect sizes range from d = {min(observed_ds.values()):.2f} to {max(observed_ds.values()):.2f}")
